# Importing the necessary libraries

In [1]:
import pandas as pd
import numpy as np
import optuna
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
print(device)

cpu


# Loading the datasets

In [ ]:
blr_df = pd.read_csv('../Data/Processed/blr_df_enhanced.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df_enhanced.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df_enhanced.csv')

In [8]:
blr_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,93.194473,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,93.578218,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,94.227231,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,93.297114,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,94.353249,33.185394
...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,95.470006,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,95.361429,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,94.854902,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,93.468912,31.310399


In [9]:
pune_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,7.100958,941.961323,0.608896,327.505071,91.114710,34.246204
1,2003-01-02,10.484796,942.675433,3.006925,290.310937,91.934631,35.734294
2,2003-01-03,12.606104,942.881457,2.872020,279.921010,92.393007,32.590960
3,2003-01-04,12.646417,943.009227,2.678422,274.952584,92.312594,35.435619
4,2003-01-05,13.066706,943.762527,3.104962,275.257170,92.569458,34.235347
...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.152634,261.457477,92.849653,33.209944
6554,2020-12-26,13.381471,942.263093,2.312996,253.686322,93.400294,31.352555
6555,2020-12-27,13.819016,941.287692,2.260313,252.095788,93.560516,33.862652
6556,2020-12-28,13.831078,940.630939,2.003774,254.229795,93.895661,32.318171


In [10]:
hyd_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,91.137446,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,92.938629,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,96.690377,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,98.016959,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,93.245268,31.878895
...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,94.673562,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,94.788311,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,94.039085,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,93.248170,31.748070


# Since we will be applying a LSTM, we will need the Date column to be Datetime column

In [11]:
blr_df['Date'] = pd.to_datetime(blr_df['Date'])
blr_df = blr_df.sort_values('Date')

In [12]:
hyd_df['Date'] = pd.to_datetime(hyd_df['Date'])
hyd_df = hyd_df.sort_values('Date')

In [13]:
pune_df['Date'] = pd.to_datetime(pune_df['Date'])
pune_df = pune_df.sort_values('Date')

# Splitting the Input and Predictor Variables

In [ ]:
features = ['DPT', 'AP', 'WS', 'WSD', 'AT', 'Albedo', 'evaporation_from_bare_soil_sum', 'evaporation_from_vegetation_transpiration_sum', 'NDVI', 'precipitation', 'surface_net_solar_radiation_sum', 'surface_thermal_radiation_downwards_sum', 'volumetric_soil_water_layer_1']
target = 'LST'

In [15]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [16]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [17]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [18]:
scaler = StandardScaler()

In [19]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Creating the sequences for RNN

In [20]:
def create_sequences(X, y, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])  # predict LST at t+1
    return np.array(X_seq), np.array(y_seq)

In [21]:
seq_len = 30

In [22]:
blr_X_seq, blr_y_seq = create_sequences(blr_X_scaled, blr_y, seq_len)
hyd_X_seq, hyd_y_seq = create_sequences(hyd_X_scaled, hyd_y, seq_len)
pune_X_seq, pune_y_seq = create_sequences(pune_X_scaled, pune_y, seq_len)

# Splitting in Training and Testing Data

In [23]:
split_idx = int(0.8 * len(blr_X_seq))
blr_X_train, blr_X_test = blr_X_seq[:split_idx], blr_X_seq[split_idx:]
blr_y_train, blr_y_test = blr_y_seq[:split_idx], blr_y_seq[split_idx:]

In [24]:
split_idx = int(0.8 * len(hyd_X_seq))
hyd_X_train, hyd_X_test = hyd_X_seq[:split_idx], hyd_X_seq[split_idx:]
hyd_y_train, hyd_y_test = hyd_y_seq[:split_idx], hyd_y_seq[split_idx:]

In [25]:
split_idx = int(0.8 * len(pune_X_seq))
pune_X_train, pune_X_test = pune_X_seq[:split_idx], pune_X_seq[split_idx:]
pune_y_train, pune_y_test = pune_y_seq[:split_idx], pune_y_seq[split_idx:]

# Converting it into Pytorch Tensor for Further Analysis

In [26]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32).to(device)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1).to(device)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32).to(device)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1).to(device)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32).to(device)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1).to(device)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32).to(device)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1).to(device)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32).to(device)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1).to(device)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32).to(device)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1).to(device)

# Defining the ANN Model

In [27]:
class LSTM_BiLSTM_Hybrid(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.2):
        super(LSTM_BiLSTM_Hybrid, self).__init__()
        self.hidden_size = hidden_size

        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                              batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)

        self.lstm1 = nn.LSTM(hidden_size * 2, hidden_size, num_layers=1,
                             batch_first=True)
        self.dropout2 = nn.Dropout(dropout)

        self.lstm2 = nn.LSTM(hidden_size, hidden_size, num_layers=1,
                             batch_first=True)
        self.dropout3 = nn.Dropout(dropout)

        self.lstm3 = nn.LSTM(hidden_size, hidden_size, num_layers=1,
                             batch_first=True)
        self.dropout4 = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.bilstm(x)
        out = self.dropout1(out)

        out, _ = self.lstm1(out)
        out = self.dropout2(out)

        out, _ = self.lstm2(out)
        out = self.dropout3(out)

        out, _ = self.lstm3(out)
        out = self.dropout4(out)

        out = self.fc(out[:, -1, :])  # last time step
        return out

# Training the Model with the Hyperparameter Tuning done using Optuna

In [28]:
def objective(trial, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor):
    hidden_size = trial.suggest_int("hidden_size", 50, 500)
    num_layers = trial.suggest_int("num_layers", 1, 2)
    dropout = trial.suggest_float("dropout", 0.0, 0.4) if num_layers > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8])
    epochs = trial.suggest_int("epochs", 50, 150)

    model = LSTM_BiLSTM_Hybrid(X_train_tensor.shape[2], hidden_size, num_layers, dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    best_loss = float('inf')
    patience = 10
    trigger_times = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_preds = model(X_test_tensor)
            val_loss = criterion(val_preds, y_test_tensor).item()

        # Early Stopping Logic
        if val_loss < best_loss:
            best_loss = val_loss
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_loss

# Applying the Model for Bengaluru

In [28]:
blr_study = optuna.create_study(direction="minimize")
blr_study.optimize(lambda trial: objective(trial, blr_X_train_tensor, blr_y_train_tensor, blr_X_test_tensor, blr_y_test_tensor), n_trials=30)

[I 2025-04-22 10:50:51,169] A new study created in memory with name: no-name-8dacf538-b253-4c61-827c-eeee162f79dd
[I 2025-04-22 10:52:26,032] Trial 0 finished with value: 23.21923828125 and parameters: {'hidden_size': 269, 'num_layers': 1, 'lr': 0.003107572977304842, 'batch_size': 8, 'epochs': 146}. Best is trial 0 with value: 23.21923828125.


Early stopping at epoch 29


[I 2025-04-22 10:55:54,794] Trial 1 finished with value: 7.426754951477051 and parameters: {'hidden_size': 338, 'num_layers': 1, 'lr': 0.0006784431546435558, 'batch_size': 8, 'epochs': 148}. Best is trial 1 with value: 7.426754951477051.


Early stopping at epoch 55


[I 2025-04-22 10:56:28,105] Trial 2 finished with value: 23.232147216796875 and parameters: {'hidden_size': 218, 'num_layers': 1, 'lr': 0.006158338391484023, 'batch_size': 8, 'epochs': 95}. Best is trial 1 with value: 7.426754951477051.


...

Early stopping at epoch 31


[I 2025-04-22 11:54:53,633] Trial 28 finished with value: 7.4346466064453125 and parameters: {'hidden_size': 54, 'num_layers': 1, 'lr': 0.0007087337988690428, 'batch_size': 8, 'epochs': 73}. Best is trial 14 with value: 7.263220310211182.


Early stopping at epoch 27


[I 2025-04-22 11:55:23,765] Trial 29 finished with value: 7.524111270904541 and parameters: {'hidden_size': 68, 'num_layers': 1, 'lr': 0.0008174375317802243, 'batch_size': 8, 'epochs': 104}. Best is trial 14 with value: 7.263220310211182.


Early stopping at epoch 23


In [29]:
blr_best_trial = blr_study.best_trial

In [30]:
print(f"Best MSE for Bengaluru : {blr_best_trial.value:.4f}")
print(f"Best Parameters for Bengaluru : {blr_best_trial.params}\n")

Best MSE for Bengaluru : 7.2632
Best Parameters for Bengaluru : {'hidden_size': 327, 'num_layers': 2, 'dropout': 0.21753723681191556, 'lr': 0.0002904506017056615, 'batch_size': 8, 'epochs': 74}



# Applying for Hyderabad

In [31]:
hyd_study = optuna.create_study(direction="minimize")
hyd_study.optimize(lambda trial: objective(trial, hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_test_tensor, hyd_y_test_tensor), n_trials=30)

[I 2025-04-22 11:55:23,810] A new study created in memory with name: no-name-0cc4e870-0720-4c00-9228-483f9ad81098
[I 2025-04-22 11:55:52,234] Trial 0 finished with value: 22.909421920776367 and parameters: {'hidden_size': 230, 'num_layers': 1, 'lr': 0.0072332754037770265, 'batch_size': 8, 'epochs': 85}. Best is trial 0 with value: 22.909421920776367.


Early stopping at epoch 11


[I 2025-04-22 11:57:58,864] Trial 1 finished with value: 9.040162086486816 and parameters: {'hidden_size': 160, 'num_layers': 2, 'dropout': 0.14196213880449782, 'lr': 0.0003742195652870773, 'batch_size': 8, 'epochs': 50}. Best is trial 1 with value: 9.040162086486816.
[I 2025-04-22 11:58:57,313] Trial 2 finished with value: 23.450485229492188 and parameters: {'hidden_size': 472, 'num_layers': 1, 'lr': 0.00025302614654230184, 'batch_size': 8, 'epochs': 74}. Best is trial 1 with value: 9.040162086486816.


Early stopping at epoch 10


[I 2025-04-22 12:00:14,826] Trial 3 finished with value: 22.785003662109375 and parameters: {'hidden_size': 266, 'num_layers': 2, 'dropout': 0.38586657997127677, 'lr': 0.002083385026541473, 'batch_size': 8, 'epochs': 64}. Best is trial 1 with value: 9.040162086486816.


...

Early stopping at epoch 35


[I 2025-04-22 12:21:55,817] Trial 28 finished with value: 22.821857452392578 and parameters: {'hidden_size': 283, 'num_layers': 2, 'dropout': 0.2760970529440806, 'lr': 0.000130733960820394, 'batch_size': 8, 'epochs': 103}. Best is trial 24 with value: 8.954193115234375.


Early stopping at epoch 11


[I 2025-04-22 12:22:48,233] Trial 29 finished with value: 22.78485679626465 and parameters: {'hidden_size': 207, 'num_layers': 1, 'lr': 0.007847243664425884, 'batch_size': 8, 'epochs': 87}. Best is trial 24 with value: 8.954193115234375.


Early stopping at epoch 22


In [32]:
hyd_best_trial = hyd_study.best_trial

In [33]:
print(f"Best MSE for Hyderabad : {hyd_best_trial.value:.4f}")
print(f"Best Parameters for Hyderabad : {hyd_best_trial.params}\n")

Best MSE for Hyderabad : 8.9542
Best Parameters for Hyderabad : {'hidden_size': 129, 'num_layers': 1, 'lr': 0.0002710091826808875, 'batch_size': 8, 'epochs': 139}



# Applying for Pune

In [34]:
pune_study = optuna.create_study(direction="minimize")
pune_study.optimize(lambda trial: objective(trial, pune_X_train_tensor, pune_y_train_tensor, pune_X_test_tensor, pune_y_test_tensor), n_trials=30)

[I 2025-04-22 12:22:48,246] A new study created in memory with name: no-name-4a007733-a9d4-44c5-870b-88a107c20237
[I 2025-04-22 12:23:37,341] Trial 0 finished with value: 45.21535110473633 and parameters: {'hidden_size': 392, 'num_layers': 1, 'lr': 0.008888147671187528, 'batch_size': 8, 'epochs': 122}. Best is trial 0 with value: 45.21535110473633.


Early stopping at epoch 11


[I 2025-04-22 12:25:24,842] Trial 1 finished with value: 45.722862243652344 and parameters: {'hidden_size': 459, 'num_layers': 1, 'lr': 0.003793898594996018, 'batch_size': 8, 'epochs': 103}. Best is trial 0 with value: 45.21535110473633.


Early stopping at epoch 18


[I 2025-04-22 12:26:19,850] Trial 2 finished with value: 45.20492172241211 and parameters: {'hidden_size': 252, 'num_layers': 2, 'dropout': 0.14555776531879538, 'lr': 0.008398451319544123, 'batch_size': 8, 'epochs': 63}. Best is trial 2 with value: 45.20492172241211.


...

Early stopping at epoch 39


[I 2025-04-22 13:05:19,122] Trial 28 finished with value: 45.37886428833008 and parameters: {'hidden_size': 52, 'num_layers': 1, 'lr': 0.00024354816228213996, 'batch_size': 8, 'epochs': 57}. Best is trial 5 with value: 9.6264009475708.


Early stopping at epoch 15


[I 2025-04-22 13:07:21,450] Trial 29 finished with value: 9.795187950134277 and parameters: {'hidden_size': 185, 'num_layers': 1, 'lr': 0.0009162953362796288, 'batch_size': 8, 'epochs': 80}. Best is trial 5 with value: 9.6264009475708.


Early stopping at epoch 54


In [35]:
pune_best_trial = pune_study.best_trial

In [36]:
print(f"Best MSE for Pune : {pune_best_trial.value:.4f}")
print(f"Best Parameters for Pune : {pune_best_trial.params}\n")

Best MSE for Pune : 9.6264
Best Parameters for Pune : {'hidden_size': 182, 'num_layers': 1, 'lr': 0.00012528369061736724, 'batch_size': 8, 'epochs': 75}



# Evaluating the Model and Visualizing the Results

In [29]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [30]:
def evaluate_predictions(y_true, y_pred):
    y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    
    # RSR = RMSE / STDEV of observed
    rsr = rmse / np.std(y_true)
    
    # PBIAS
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse,
        "RSR": rsr,
        "PBIAS": pbias
    }

In [31]:
def plot_predictions(y_true, y_pred, title="Prediction vs Ground Truth"):
    plt.figure(figsize=(10, 5))
    plt.plot(y_true.squeeze(), label="True", alpha=0.7)
    plt.plot(y_pred.squeeze(), label="Predicted", alpha=0.7)
    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel("LST")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [32]:
from optuna.visualization.matplotlib import plot_optimization_history

def plot_optuna_loss(study, title="Optuna Loss Over Trials"):
    fig = plot_optimization_history(study)
    fig.gca().set_title(title)
    plt.tight_layout()
    plt.show()

In [33]:
def train_final_model(X_train, y_train, X_test, y_test, best_params):
    model = LSTM_BiLSTM_Hybrid(
        input_size=X_train.shape[2],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params.get("dropout", 0.0)
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_train, y_train)
    loader = torch.utils.data.DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model.train()
    for epoch in range(best_params["epochs"]):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    return model

In [34]:
import plotly.graph_objects as go
def plot_predictions_plotly(y_true, y_pred, title="Prediction vs Ground Truth (Test Data)"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=y_true.squeeze(),
        mode='lines',
        name='True',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        y=y_pred.squeeze(),
        mode='lines',
        name='Predicted',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Time Step (Test Set)",
        yaxis_title="LST",
        legend=dict(x=0.01, y=0.99),
        template='plotly_white',
        height=500,
        width=1000
    )

    fig.show()

In [35]:
blr_best_params   = {'hidden_size': 327, 'num_layers': 2, 'dropout': 0.21753723681191556, 'lr': 0.0002904506017056615, 'batch_size': 8, 'epochs': 74}
pune_best_params  = {'hidden_size': 129, 'num_layers': 1, 'lr': 0.0002710091826808875, 'batch_size': 8, 'epochs': 139}
hyd_best_params   = {'hidden_size': 182, 'num_layers': 1, 'lr': 0.00012528369061736724, 'batch_size': 8, 'epochs': 75}

In [36]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

KeyboardInterrupt: 

In [ ]:
blr_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_blr = model(blr_X_test_tensor).cpu().numpy()
    y_true_blr = blr_y_test_tensor.cpu().numpy()

In [ ]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [ ]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [ ]:
hyd_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_hyd = model(hyd_X_test_tensor).cpu().numpy()
    y_true_hyd = hyd_y_test_tensor.cpu().numpy()

In [ ]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [ ]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [ ]:
pune_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_pune = model(pune_X_test_tensor).cpu().numpy()
    y_true_pune = pune_y_test_tensor.cpu().numpy()

In [ ]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [ ]:
torch.save(blr_model.state_dict(), "blr_lstm_bilstm_hybrid_model.pth")

In [ ]:
torch.save(hyd_model.state_dict(), "hyd_lstm_bilstm_hybrid_model.pth")

In [ ]:
torch.save(pune_model.state_dict(), "pune_lstm_bilstm_hybrid_model.pth")